# Módulo 1 · ADK Code-First + A2A
**Caso:** Supervisor de Finanzas que delega a un Especialista en Presupuestos, manteniendo el estado de la conversación.

## Parte 0 · Preparación (10 min)

In [ ]:
import os, sys, subprocess, asyncio, time
from pathlib import Path

# 1) Ubica la raíz del proyecto y prepara los imports
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "config.py").exists())
for ruta in (RAIZ, RAIZ / "modulo1_adk_a2a" / "src"):
    sys.path.insert(0, str(ruta))
os.chdir(RAIZ)

# 2) Variables de entorno (.env) y datos de ejemplo
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")
if not (RAIZ / "data" / "finanzas.db").exists():
    subprocess.run([sys.executable, "data/seed_data.py"], check=True)

import logging, warnings
logging.getLogger("google_adk").setLevel(logging.ERROR)     # menos ruido de logs de ADK
warnings.filterwarnings("ignore", category=UserWarning)     # avisos EXPERIMENTAL de ADK (RemoteA2aAgent, etc.)

import google.adk
print("RAIZ:", RAIZ)
print("google-adk:", google.adk.__version__)

### Modelo: Gemini en modo enterprise

In [ ]:
from finanzas_agents.eventos import preguntar, estado 
from shared.config import MODEL_NAME

# Credenciales de gcloud.
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

proyecto = os.getenv("GOOGLE_CLOUD_PROJECT", "")
if not proyecto or proyecto == "tu-proyecto":
    raise RuntimeError("Falta GOOGLE_CLOUD_PROJECT en tu .env: pon el ID de tu proyecto (no 'tu-proyecto').")

import google.auth
try:
    google.auth.default() 
except google.auth.exceptions.DefaultCredentialsError:
    raise RuntimeError("No hay credenciales. En una terminal ejecuta: gcloud auth application-default login")

MODELO = MODEL_NAME
os.environ["MODEL_NAME"] = MODEL_NAME
print("Proyecto:", proyecto, "| Región:", os.environ["GOOGLE_CLOUD_LOCATION"], "| Modelo:", MODELO)

## Parte 1 · Herramienta tipada y contratos 
Una **herramienta** es una función Python con tipos y *docstring*: el modelo lee el docstring para decidir cuándo y cómo llamarla.

In [ ]:
from finanzas_agents.tools import consultar_presupuesto
import inspect
print(inspect.signature(consultar_presupuesto))
print(inspect.getdoc(consultar_presupuesto)[:200], "...\n")

r = consultar_presupuesto("Norte", "2026-Q3")
print(r["status"], r["division"], r["trimestre"], "→ desviación", r["desviacion_pct"], "%")
for l in r["lineas"]:
    print(f"  {l['categoria']:12} presup {l['presupuestado']:>9,.0f}  ejec {l['ejecutado']:>9,.0f}")

**Ejercicio 1.** Consulta la división **Sur**, trimestre **2026-Q1**, solo la categoría **Tecnologia**, e imprime su desviación.

In [ ]:
# TODO: llama a consultar_presupuesto con los 3 argumentos (division, trimestre, categoria)
r1 = ...
print(r1)

**Contratos Pydantic:** la validación estricta ocurre *antes* del LLM. Una petición inválida se rechaza por código, no por criterio del modelo.

In [ ]:
from pydantic import ValidationError
from finanzas_agents.contracts import BudgetRequest

print(BudgetRequest(division="Norte", trimestre="2026-Q3"))
try:
    BudgetRequest(division="Marte", trimestre="2026-Q3")
except ValidationError as e:
    print("❌ rechazado:", e.errors()[0]["msg"])

**Ejercicio 2.** Demuestra que el trimestre **`2026-Q5`** también se rechaza e imprime el mensaje de error.

In [ ]:
# TODO: intenta crear un BudgetRequest con trimestre "2026-Q5" dentro de try/except ValidationError
...

## Parte 2 · Tu primer agente
Un agente = **modelo + instrucción + herramientas**. `description` = *qué es* (la lee el router); `instruction` = *cómo* cumple su objetivo.

In [ ]:
from google.adk.agents import Agent

presupuestos_agent = Agent(
    name="especialista_presupuestos",          # sin espacios
    model=MODELO,
    description="TODO: una frase que diga QUÉ hace este agente (la leerá el supervisor)",
    instruction="TODO: reglas: usar SIEMPRE la herramienta, nunca inventar cifras, qué reportar...",
    tools=[consultar_presupuesto],
)

In [ ]:
# Ejecuta el agente. Observa los eventos: 🔧 llamada a herramienta → 📥 resultado → 🤖 respuesta final
await preguntar(presupuestos_agent, "¿Cómo va la división Norte en 2026-Q3?", session_id="p2")

## Parte 3 · Supervisor y delegación
El **Supervisor** no consulta presupuestos: decide a quién delegar usando las `description` de sus `sub_agents` (hand-off jerárquico).

In [ ]:
supervisor = Agent(
    name="supervisor_finanzas",
    model=MODELO,
    description="Orquestador financiero que delega a especialistas.",
    instruction="TODO: 'Eres el Supervisor de Finanzas. Delega ...'  (menciona a qué especialista y que conserve el contexto)",
    sub_agents=[],   # TODO: agrega el especialista de presupuestos
)

In [ ]:
await preguntar(supervisor, "¿Cómo va la división Norte en 2026-Q3?", session_id="s2")

## Parte 4 · Estado de la conversación
La memoria de corto plazo vive en la **sesión**, no en el prompt. Misma `session_id` → misma conversación.

In [ ]:
# Seguimiento en la MISMA sesión: no repetimos la división
await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="s2")

In [ ]:
# Otra sesión = otra conversación: no hay contexto previo
await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="s3")

## Parte 5 · A2A: del handshake al servidor real

### 5.1 Protocolo sin red ni LLM

In [ ]:
from finanzas_agents.a2a_local import Supervisor, BudgetWorker

sup_local = Supervisor(workers={"especialista_presupuestos": BudgetWorker()})
print("handshake →", sup_local.handshake("especialista_presupuestos"))
print(sup_local.ask("sesion1", "¿Cómo va la división Norte en Q3?"))
print(sup_local.ask("sesion1", "¿Y en Q2?"))
print("estado →", sup_local.sessions)

### 5.2 A2A real por HTTP
Levantamos el especialista como **servidor A2A** (puerto 8001) y el supervisor lo consume como agente remoto.

In [ ]:
import httpx

entorno = {**os.environ,
           "PYTHONPATH": os.pathsep.join([str(RAIZ), str(RAIZ / "modulo1_adk_a2a" / "src")])}
servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "finanzas_agents.a2a_server:a2a_app", "--port", "8001"],
    env=entorno, cwd=RAIZ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

URL_CARD = "http://localhost:8001/.well-known/agent-card.json"
card = None
for _ in range(60):
    try:
        card = httpx.get(URL_CARD, timeout=2).json()
        break
    except Exception:
        time.sleep(0.5)
assert card, "El servidor A2A no arrancó: revisa el puerto 8001"
print("Agent card →", card["name"], "|", card["description"][:70], "...")

In [ ]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

especialista_remoto = RemoteA2aAgent(
    name="especialista_presupuestos",
    description="Especialista de presupuestos expuesto vía A2A.",
    agent_card=...,   # TODO: la URL de la agent card (URL_CARD)
)
supervisor_remoto = Agent(
    name="supervisor_remoto",
    model=MODELO,
    description="Supervisor que delega por red.",
    instruction="Eres el Supervisor de Finanzas. Delega las consultas de presupuesto al especialista remoto.",
    sub_agents=[...],   # TODO: el especialista remoto
)

In [ ]:
await preguntar(supervisor_remoto, "¿Cómo va la división Norte en 2026-Q3?", session_id="a2a1")

### 5.3 Experimento: ¿el estado también viaja por A2A?

In [ ]:
# Seguimiento por A2A: MISMA sesión que la pregunta anterior (a2a1)
await preguntar(supervisor_remoto, "¿Y en 2026-Q2?", session_id="a2a1")

In [ ]:
# Control: sesión NUEVA (no debería tener contexto y pedir la división)
await preguntar(supervisor_remoto, "¿Y en 2026-Q2?", session_id="a2a2")

### 5.4 Reto opcional: router con **dos** especialistas remotos (10 min)
Hasta aquí el supervisor tenía **un** especialista remoto. Ahora publicamos **Tesorería** como segundo servidor A2A (puerto **8002**) y el supervisor decide a cuál transferir según el dominio de la pregunta.

> Recuerda: un agente remoto es una **caja negra** para el supervisor. Quien decide a quién pasar el control es **el supervisor**, no los especialistas entre sí (a diferencia del hand-off **local** de la Parte 7).
> Ejecútalo **antes de cerrar el servidor de la 5.2** (la celda de `servidor.terminate()`).

In [ ]:
# Segundo servidor A2A: Tesorería en el puerto 8002 (el de Presupuestos sigue en el 8001)
servidor_tes = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "finanzas_agents.a2a_server:a2a_app_tesoreria", "--port", "8002"],
    env=entorno, cwd=RAIZ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

URL_CARD_TES = "http://localhost:8002/.well-known/agent-card.json"
card_tes = None
for _ in range(60):
    try:
        card_tes = httpx.get(URL_CARD_TES, timeout=2).json()
        break
    except Exception:
        time.sleep(0.5)
assert card_tes, "El servidor de Tesorería no arrancó: revisa el puerto 8002"
print("Agent card →", card_tes["name"], "|", card_tes["description"][:70])

In [ ]:
# Un RemoteA2aAgent por especialista (un agente solo puede tener UN padre: no reutilices los de la 5.2)
presupuestos_rem = RemoteA2aAgent(name="especialista_presupuestos",
                                  description="Presupuesto vs ejecutado por división y trimestre.",
                                  agent_card=URL_CARD)
tesoreria_rem = RemoteA2aAgent(name="especialista_tesoreria",
                               description="Flujo de caja, liquidez y pagos.",
                               agent_card=...)   # TODO: la URL de la agent card de Tesorería (URL_CARD_TES)

supervisor_router = Agent(
    name="supervisor_router",
    model=MODELO,
    description="Supervisor que enruta por red a dos especialistas remotos.",
    instruction="TODO: 'Eres el Supervisor de Finanzas. Presupuesto -> ...; liquidez -> ...'  (menciona a los dos especialistas por su name)",
    sub_agents=...,   # TODO: la lista con los DOS especialistas remotos
)

In [ ]:
# Dos preguntas de dominios distintos en la MISMA sesión
await preguntar(supervisor_router, "¿Cuál es la posición de liquidez?", session_id="r1")
await preguntar(supervisor_router, "¿y el presupuesto de Norte en 2026-Q3?", session_id="r1")

In [ ]:
# Cierra el servidor de Tesorería
servidor_tes.terminate()
print("servidor de Tesorería detenido")

In [ ]:
# Cierra el servidor cuando termines la Parte 5 (o déjalo si vas a repetir la celda anterior)
servidor.terminate()
print("servidor A2A detenido")

## Parte 6 · Workflows y `output_key`

In [ ]:
from google.adk.agents import SequentialAgent

analista = Agent(name="analista_presupuesto", model=MODELO,
                 description="Consulta presupuesto vs ejecutado.",
                 instruction=("Usa consultar_presupuesto para la división y trimestre pedidos. Resume totales y desviación %. "
                              "Reporta las cifras EXACTAMENTE como las devuelve la herramienta: mismas unidades, "
                              "sin multiplicarlas, convertirlas ni redondearlas."),
                 tools=[consultar_presupuesto], output_key="analisis_presupuesto")
redactor = Agent(name="redactor_ejecutivo", model=MODELO,
                 description="Redacta resúmenes ejecutivos.",
                 instruction=("Redacta un resumen ejecutivo de 3 líneas basado SOLO en este análisis: {analisis_presupuesto}. "
                              "Copia las cifras tal cual aparecen en el análisis, sin cambiar sus unidades."),
                 output_key="resumen_ejecutivo")
pipeline = SequentialAgent(name="pipeline_presupuesto", sub_agents=[analista, redactor])

await preguntar(pipeline, "Norte 2026-Q3", session_id="w2")
print("\nEstado de la sesión →", list((await estado(pipeline, "w2")).keys()))

**Ejercicio 3.** Crea un `LoopAgent` llamado `refinador` que repita **3 veces** un agente `revisor` (sin `model`, sin `instruction` en el LoopAgent) e imprime su `max_iterations`.

In [ ]:
from google.adk.agents import LoopAgent

revisor = Agent(name="revisor", model=MODELO, description="Mejora el resumen.",
                instruction="Mejora la claridad de: {resumen_ejecutivo}. Mantén las cifras exactas.",
                output_key="resumen_ejecutivo")
# TODO: refinador = LoopAgent(name=..., sub_agents=[...], max_iterations=...)
refinador = ...
print(refinador.max_iterations)

## Parte 7 · Reto: hand-off jerárquico a Tesorería (10 min)

In [ ]:
# TODO 1: crea especialista_tesoreria (description: flujo de caja, liquidez y pagos; instruction: responde solo sobre tesorería
#         y, si preguntan por presupuesto, devuelve el control)
tesoreria_agent = ...

# TODO 2: crea supervisor_v2 con sub_agents=[presupuestos_agent.clone(), tesoreria_agent]   (¡clone!)
supervisor_v2 = ...

# TODO 3: pregunta por liquidez y después por el presupuesto de Norte, en la MISMA sesión

## Parte 8 · Autoevaluación del entregable (10 min)
Ejecuta la celda: debes ver ✅ en todo. Si algo marca ❌, vuelve a la parte indicada.

In [ ]:
def check(nombre, cond, parte):
    print(("✅" if cond else "❌ (revisa Parte %s)" % parte), nombre)
    return cond

resultados = []

# Contratos
try:
    BudgetRequest(division="Marte", trimestre="2026-Q3"); rechaza = False
except ValidationError:
    rechaza = True
resultados.append(check("Un contrato inválido se rechaza por código, no por el LLM", rechaza, 1))

# Herramienta
t = consultar_presupuesto("Norte", "2026-Q3")
resultados.append(check("La herramienta devuelve Norte 2026-Q3 con exceso > 15 %", t["status"] == "ok" and t["desviacion_pct"] > 15, 1))

# Delegación + estado (sesión nueva y limpia)
r1 = await preguntar(supervisor, "¿Cómo va la división Norte en 2026-Q3?", session_id="eval", verbose=False)
resultados.append(check("El supervisor delega y responde con la cifra (25.3 % / 380,900)",
                        any(x in r1 for x in ("25.3", "380,900", "380900")), 3))
r2 = await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="eval", verbose=False)
# Gemini no es determinista: lo que NO debe pasar es que vuelva a pedir la DIVISIÓN (el contexto se perdió).
pide_division = "divisi" in r2.lower() and "norte" not in r2.lower()
resultados.append(check("El seguimiento conserva la división (no la vuelve a pedir)", bool(r2) and not pide_division, 4))

# A2A
resultados.append(check("La agent card A2A se publicó", bool(card) and card.get("name") == "especialista_presupuestos", 5))

print("\n🎓 ENTREGABLE LISTO" if all(resultados) else "\n⚠️ Falta algo: revisa los ❌")

**Siguiente módulo:** RAG financiero con reranking (K=20 → N=3).